<a href="https://colab.research.google.com/github/rishh19/FlyRank-AI-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/rishh19/FlyRank-AI-Internship"
REPO_DIR = "FlyRank-AI-Internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True,
        )
    os.chdir(REPO_DIR)

print("Current directory:", os.getcwd())

Current directory: /content/FlyRank-AI-Internship


In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded successfully.")

HF_TOKEN loaded successfully.


In [3]:
!pip -q install duckdb huggingface_hub pandas pyarrow scikit-learn

In [6]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN
)

print(path)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [7]:
import duckdb

con = duckdb.connect()

con.sql(f"""
SELECT *
FROM read_parquet('{path}')
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rishh19/FlyRank-AI-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two Paper Findings and My Methodology Questions

### Finding 1
The research reports that AI-assisted SEO can improve search performance.

**Methodology Question:**  
How was the target label defined, and how was improvement measured across different websites?

### Finding 2
The paper reports that machine learning models can prioritize content for optimization.

**Methodology Question:**  
Was the validation performed using an independent test set or a time-aware split to avoid optimistic performance estimates?

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
paper_findings = {
    "Finding 1": "AI-assisted SEO can improve search performance.",
    "Finding 2": "Machine learning can prioritize content for optimization."
}

for finding, text in paper_findings.items():
    print(f"{finding}: {text}")

Finding 1: AI-assisted SEO can improve search performance.
Finding 2: Machine learning can prioritize content for optimization.


## 2. My Model Under an Honest Split

I re-evaluated the Decision Tree model using the same feature set but with a separate training and testing split. This provides a more honest estimate of model performance compared to evaluating on the training data. The comparison is intended for decision-support and not as proof of real-world performance.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

query = f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM read_parquet('{path}')
LIMIT 5000
"""

data = con.sql(query).df()

data["target"] = (
    data["gsc_impressions"] > data["gsc_impressions"].median()
).astype(int)

X = data[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
    ]
]

y = data["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

model = DecisionTreeClassifier(random_state=42)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

before_accuracy = 0.70
after_accuracy = accuracy_score(y_test, predictions)

comparison = pd.DataFrame({
    "Evaluation": ["Previous Baseline", "Honest Split"],
    "Accuracy": [before_accuracy, round(after_accuracy, 3)]
})

comparison

,Evaluation,Accuracy
0,Previous Baseline,0.7
1,Honest Split,1.0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.